In [43]:
import sys
import json
import uuid
import random
import trimesh

from hra_amap.registration.tissue import TissueBlock
from hra_amap.registration.dataclass import Projection
from hra_amap.registration.rui import RUIProcessor
from hra_amap.cli.registration_stage_2 import ProjectionBlockGenerator

from copy import deepcopy
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

In [44]:
stage_1_projection_Path = Path("../raw-data/millitome/fallopian-tube-female-right-upenn/v1.0/projections.pickle.gz")
config_path = Path("../input-data/millitome/fallopian-tube-female-right-upenn/v1.0/config.yaml")
projected_blocks = ProjectionBlockGenerator(stage_1_projection_Path, config_path).generate_projections()

In [46]:
output_dir = Path("../output-data/millitome/fallopian-tube-female-right-upenn/v1.0")
processor = RUIProcessor(blocks=projected_blocks, registration_dir=output_dir)
processor.initialize_registration()
processor.generate_rui_locations(config_path)

In [47]:
# create scene with the projected tissue blocks
trimesh.Scene([projected_blocks]).show()

In [48]:
projected_blocks_aabb = [block.bounding_box for block in deepcopy(projected_blocks)]

# ensure same colors on the axis-aligned bounding boxes
for index, aabb in enumerate(projected_blocks_aabb):
    aabb.visual.vertex_colors = projected_blocks[index].visual.vertex_colors[0]

In [49]:
# create scene with the projected tissue blocks (EUI view)
trimesh.Scene([projected_blocks_aabb]).show()

In [ ]:
scene = trimesh.Scene([projected_blocks])

# Export the scene to GLB format
glb_data = scene.export(file_type='glb')

# Write to a .glb file
with open('../output-data/millitome/fallopian-tube-female-right-upenn/v1.0/fallopian-tube-female-right-upenn.glb', 'wb') as f:
    f.write(glb_data)

In [ ]:
 with open(output_dir / 'rui_locations.jsonld', 'r') as f:
    jsonld = json.load(f)

extraction_sites = [sample['rui_location'] for donor in jsonld['@graph'] for sample in donor['samples']]

with open(output_dir / 'dataset-graph.jsonld', 'w') as f:
    json.dump(jsonld, f, indent=2)

with open(output_dir / 'extraction-sites.jsonld', 'w') as f:
    json.dump(extraction_sites, f, indent=2)